# Imports set up rendering

In [1]:
import sys
from pathlib import Path

# Add project root to Python path FIRST to ensure our modules take priority
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Added {project_root} to Python path")
elif sys.path.index(str(project_root)) != 0:
    # Move to front if it exists but isn't first
    sys.path.remove(str(project_root))
    sys.path.insert(0, str(project_root))
    print(f"Moved {project_root} to front of Python path")

# import os
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ['MUJOCO_GL'] = 'glfw'
# os.environ['PYOPENGL_PLATFORM'] = 'glfw'
# os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 0

import os, sys

if sys.platform == "darwin":  # macOS
    # MuJoCo: EGL is not valid on macOS
    os.environ["MUJOCO_GL"] = "glfw"

    # Don't set PyOpenGL EGL platform on macOS
    os.environ.pop("PYOPENGL_PLATFORM", None)

    # JAX: no CUDA on Mac
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ.pop("XLA_FLAGS", None)

    # Force CPU backend (safe default)
    os.environ["JAX_PLATFORM_NAME"] = "cpu"

else:
    # (Optional) keep your Linux settings here if you also run on a GPU server
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"
    os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"


# JAX setup
import jax
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# Note: jax_persistent_cache_enable_xla_caches may not be available in all JAX versions
try:
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
except AttributeError:
    pass  # Skip if not available in this JAX version
jax.config.update("jax_default_matmul_precision", "high")

# Matplotlib setup
import matplotlib as mpl
mpl.rcParams.update({
    'font.size': 10,
    'axes.linewidth': 2,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'pdf.fonttype': 42,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'pdf.use14corefonts': True,
    'svg.fonttype': 'none',
    'font.family': 'sans-serif',
    'font.serif': 'Arial',
})

Added /Users/johnsonr/src/3d_tracking_dataset to Python path


In [2]:
%load_ext autoreload
%autoreload 2

import hydra
import ray
import json
import mujoco
import numpy as np
import pandas as pd
import mediapy as media
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm
from omegaconf import DictConfig, OmegaConf
from pathlib import Path
from utils.path_utils import convert_dict_to_path, convert_dict_to_string, save_config
from utils.optimized_floor_alignment import * 
from utils.kp_viz import *
from utils.io import *
import utils.io_dict_to_hdf5 as ioh5



✅ Ground contact alignment with JIT optimization integrated


In [3]:
base_path = Path('/Users/johnsonr/datasets') ##### Change this to your dataset location
data_path = base_path / 'courtship' /'data3D.csv'
skeleton_path = base_path / 'courtship' / 'courtship_13.json'
flybody_path = project_root /'assets/fruitfly_v1/fruitfly_v1_free.xml'
floor_path = project_root / 'assets/fruitfly_v1/floor.xml'

In [4]:
stac_data = ioh5.load(data_path.parent/f'ik_output_0.h5', enable_jax=True)
stac_data.keys()

dict_keys(['egocentric_site_names', 'egocentric_site_pos', 'kp_data', 'kp_names', 'marker_sites', 'names_qpos', 'names_xpos', 'offsets', 'qpos', 'qvel', 'xpos', 'xquat'])

In [5]:
def make_vidoes(
    mj_model,
    mj_data,
    qposes_rollout,
    scene_option,
    camera="track1",
    height=512,
    width=512,
):
    """
    Make a video of the rollout and reference superimposed.
    """
    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for t in tqdm(range(len(qposes_rollout))):
            mj_data.qpos = qposes_rollout[t]
            mujoco.mj_forward(mj_model, mj_data)
            
            renderer.update_scene(
                mj_data, camera=f"{camera}", scene_option=scene_option
            )
            renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = False
            pixels = renderer.render()
            frames.append(pixels)
    return frames

In [6]:
# mj_model = env.mj_model
spec = mujoco.MjSpec().from_file(flybody_path.as_posix())
floor_spec = mujoco.MjSpec().from_file(floor_path.as_posix())
spawn_frame = floor_spec.worldbody.add_frame(
                pos=[0,0,-.125],
                quat=[1,0,0,0],
            )
spawn_body = spawn_frame.attach_body(spec.body("thorax"), "", suffix='_fly')
mj_model = floor_spec.compile()
mj_data = mujoco.MjData(mj_model)

t = 0
frames=[]
qpos_traj =  mocap_qpos.copy()
# Set up rendering
height, width = 448, 448
camera = mj_model.camera(0).name
scene_option = mujoco.MjvOption()
scene_option.geomgroup[:] = [1, 1, 1, 0, 0, 0]
scene_option.sitegroup[:] = [1, 1, 1, 1, 1, 0]
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False

# Render from multiple camera angles
all_frames = []
for cam in [1,2,7]:
    camera = mj_model.camera(cam).name
    frames = make_vidoes(mj_model, mujoco.MjData(mj_model), qpos_traj, scene_option, camera=camera, height=height, width=width)
    all_frames.append(frames)
all_frames = np.concatenate(all_frames, axis=1)
media.show_video(all_frames, fps=30)
# media.show_video(frames, fps=30)

NameError: name 'mocap_qpos' is not defined